In [ ]:
import coffea
import coffea.hist as hist
import uproot
import matplotlib.pyplot as plt
import mplhep as hep
import numpy as np

import awkward as ak

from sklearn.cluster import DBSCAN

from coffea.nanoevents import NanoEventsFactory, NanoAODSchema,BaseSchema

In [ ]:
from collections import OrderedDict

# Load ntuples
Here we will load ntuples for signal and data (high-MET skim), where the clusters are already reconstructed and stored in the ntuples.

We will use these ntuples to plot some of the key variables that will used to veto background clusters

In [ ]:
fpath={}
tree ={}


# load high-MET skim data
path = '/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/data/'
fpath['data'] = path + 'Run2_displacedJetMuonNtupler_V1p17_Data2016_Data2017_Data2018-HighMET_goodLumi.root'

mass = [40]
OLD_CTAU = np.array([1000, 10000, 100000])#in mm

path = '/eos/uscms/store/user/cmsdas/2025/long_exercises/MDS/signal/nocuts/'

for m in mass:
    for ct in OLD_CTAU:
        key = 'MC_'+str(m)+'_'+str(ct)                       
        fpath[key] = path +  'ggH_HToSSTobbbb_MH-125_MS-40_ctau-'+str(ct)+'_137000pb_weighted.root'
        
        
for k,v in fpath.items():
    tree[k] = NanoEventsFactory.from_root(v,
                                   schemaclass=BaseSchema,
                                   treepath="MuonSystem"
                                  ).events()

In [ ]:
# See what's contained in the tree dictionary
tree

# Trigger selection

Trigger selection is the beginning of any CMS analysis. 

We are using `MET > 200 GeV` due to the lack of dedicated trigger on the MDS object. 

As a result, in order for signal events to pass the large MET trigger
 - the Higgs is recoiled against a high pT jet
 - When the LLPs decays beyond calorimeters, the decay
 - Since the Higgs has a high pT, the MET will be large
 
You will try to confirm this for yourself first.

 - Use the `met` branch to access the reconstructed MET
 - Use the `gHiggsPt` branch to access the generated Higgs pT  

# Ex: Plot the MET and Higgs pT for signals

 - Focus on the region beyond 200 GeV
 - Q: What do you observe?

In [ ]:
h= hist.Hist("Events",
              hist.Cat("var","Variable"), 
              hist.Bin("pT", "MET/Higgs pT[GeV]", 50, 200, 1200)
             )

sample = "MC_40_1000"
# Fill the code
h.fill(var = "MET", pT = tree[sample].met )
h.fill(var = "Higgs pT", pT = tree[sample].gHiggsPt )
        
ax = hist.plot1d(h,density=True)
ax.legend(title = sample)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

## Solution: MET and Higgs pT

Both variables are filled into the same histogram. The `var` category keeps the MET and the Higgs pT as separate curves, and `density=True` normalizes each one so that the shapes can be compared.

Physics: a large MET in the signal comes from the recoil against the Higgs, which is boosted by the ISR jet. The MET and Higgs pT should therefore look alike above 200 GeV.

Pitfall: both `met` and `gHiggsPt` hold one value per event, so they can be filled directly. Events below 200 GeV go to the underflow bin and are not shown.

# Ex: Trigger efficiency for signals

In [ ]:
plt.style.use(hep.style.CMS)  

In [ ]:
h_MET= hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("met", "MET[GeV]", 50, 0, 500)
             )

for m in mass:
    for ct in OLD_CTAU:
        # Fill the histogram h_MET with all the signals
        sample = 'MC_'+str(m)+'_'+str(ct)
        h_MET.fill(sample=sample, met=tree[sample].met)
        
ax = hist.plot1d(h_MET,density=True)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

## Solution: MET shape per signal sample

Each lifetime is filled into the same histogram under its own `sample` category, so one plot holds all the samples. The `sample` string must match the key in `tree` exactly, otherwise the entries land in a different category.

Physics: compare the shapes across lifetimes. A decay beyond the calorimeter leaves the LLP momentum out of the calorimeter sum, which adds to the MET, so the curves are expected to differ.

Pitfall: `sample` is set inside the loop, so reset it there. Use keyword arguments in `fill` so the axes cannot be swapped by mistake.

# Q: What is the signal efficiencies for MET>200 GeV?

 - Use the following function `integral` the dense axes from `thres_lo` to infinity
 - Use the function `integral_low` the dense axes from lower bound to thres_high (for later parts)

In [ ]:
#  Integral the dense axes from thres_lo to infinity
def integral(h,sample,thres_lo):
    
    name = h.dense_axes()[0].name
    
    numer = h.integrate("sample",sample).integrate(name,slice(thres_lo,None),overflow="over").values()[()]
    denom = h.integrate("sample",sample).sum(name,overflow="over").values()[()]    
    
    return numer/denom

# Integral the dense axes from lower bound to thres_high
def integral_low(h,sample,thres_high):
    
    name = h.dense_axes()[0].name

    numer = h.integrate("sample",sample).integrate(name,slice(None,thres_high)).values()[()]
    denom = h.integrate("sample",sample).sum(name,overflow="over").values()[()]    

    return numer/denom

## Solution: signal efficiency for MET > 200 GeV

`integral(h, sample, thres_lo)` returns the fraction of events above the threshold. The numerator integrates from 200 GeV to infinity (`overflow="over"` keeps the events beyond the last edge), and the denominator sums every event of that sample, including those in the overflow bin.

Result: the MET trigger efficiency for each signal lifetime. This number feeds the signal yield.

Pitfall: the threshold has to sit on a bin edge. With 50 bins over 0-500 GeV (10 GeV wide), 200 GeV is an edge, so the cut is exact. A threshold inside a bin would only be approximated.

In [ ]:
# Choose from the following bin edges
h_MET.axis("met").edges()

In [ ]:
for m in mass:
    for ct in OLD_CTAU:
        sample = 'MC_'+str(m)+'_'+str(ct) 
        # Fill the code
        eff = integral(h_MET, sample, 200)
        print("sample = %s, eff(MET>200) = %.5f "%(sample, eff))

# Event selections

The event selections for this search are kept at minimal to be as model independent as possible. 

The full event-level selections are 
 - MET trigger 
 - offline MET > 200 GeV
 - MET filters (make sure MET is not badly reconstructed)

For the ISR jet, the selections are
 - pT > 30 GeV
 - abs($\eta$) < 2.4
 - Pass tight jet Id

In [ ]:
def select_events(events,isData=False):
    
    ########### SELECTION: CLUSTERS ############
    ### At least 1 cluster (decay in CSC and matched to LLP for signal)
    ###
    if isData:
        sel_clusters = (events.cscRechitClusterSize)>0
    else:
        sel_clusters = (events.cscRechitCluster_match_gLLP==True) & (events.cscRechitCluster_match_gLLP_csc==True)

    ########### SELECTION: JETS ############
    sel_jet = (events.jetPt > 30) &\
              (abs(events.jetEta)<2.4) &\
              (events.jetTightPassId==True)
    
    ########### SELECTION: events ############
    # at least 1 selected cluster and jets
    # and pass MET selections
    sel_ev =  (events.METTrigger==True) &\
              (events.met>200) &\
              (events.metFilters==True) &\
              (ak.sum(sel_jet,axis=1)>=1) &\
              (ak.sum(sel_clusters,axis=1)>0)   
    
    return sel_clusters, sel_jet, sel_ev

# Cluster level selections

Aside from the MDS shower, there are not many handles in the events to reject background events.

Afterall, we only expect the shower, MET and an ISR jet in the signal events. 

Therefore, we will focus on using the MDS object features to enhance the signal to background ratio.  

First, we need to remove objects from the SM that could also create showers in the muon system. 

 - Punch-through jets
 - high pT muons


The dominant background from the main collision comes from punch-through jets that are not fully contained in the calorimeters and high pT muons that could create bremsstrahlung showers in the muon detectors. 

To remove those background, we reject clusters by matching them to reconstructed jets and muon. 

The pT of the jet/muon that the cluster is matched to are saved in objects called `cscRechitClusterJetVetoPt` 
and `cscRechitClusterMuonVetoPt`. 

In this exercise you will plot the two variables for signal and background and apply a selection on the two variables to remove clusters from jets and muons.

# Ex1 - plot jet veto and muon veto

 - Access the jet veto pT with `cscRechitClusterJetVetoPt`
 - Access the muon veto pT with `cscRechitClusterMuonVeto`

In [ ]:
nBins    = 50
bin_lo   = 0 
bin_high = 500 

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("pt", "Jet veto pT[GeV]", nBins, bin_lo, bin_high)
             )

for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # get the Jet veto Pt
    VetoPt = tree[sample].cscRechitClusterJetVetoPt
    
    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    #
    # Apply the cluster and event selections onto VetoPt
    VetoPt = VetoPt[sel_clusters][sel_ev]
    
    # Select the VetoPt of the largest cluster in the event
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    VetoPt = ak.fill_none(ak.firsts(VetoPt,axis=1),-1 ) 
    
    h.fill(sample=sample,pt=VetoPt)

## Solution: jet veto pT distribution

Each cluster carries one jet veto pT value, so the cluster mask is applied first (`VetoPt[sel_clusters]`) and then the event mask (`[sel_ev]`). Taking `ak.firsts` keeps the first cluster per event, and `fill_none(..., -1)` puts events without a surviving cluster into the underflow bin.

Units: GeV. Binning: 50 bins from 0 to 500 GeV, so each bin is 10 GeV wide and the 10 GeV threshold used later is a bin edge.

Pitfall: apply the cluster mask before the event mask. Applying the event mask first would select the wrong set of events for the per-cluster array.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-3,0.1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies Jet veto pT < 10 GeV?

In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 10)
    print("sample = %s, eff(VetoPT<10) = %.5f "%(sample, eff))

## Solution: jet veto efficiency (pT < 10 GeV)

`integral_low(h, sample, 10)` is the fraction of clusters whose jet veto pT is below 10 GeV, i.e. the clusters that survive the jet veto. Events without a cluster are in underflow and are excluded by the helper, so the number is conditional on having a cluster. The 10 GeV edge is on a bin boundary.

Physics: the jet veto pT is the pT of the reconstructed jet the cluster is matched to. A punch-through jet deposits energy that is not fully absorbed, so a matched jet with high pT signals a background cluster. Signal clusters are not matched to jets, so their veto pT is small.

Pitfall: a small veto value does not mean the jet is missing from the event. It only means no jet is matched to this cluster. Check the matching definition in the ntuple if the numbers look odd.

# Plot muon veto 

In [ ]:
# Think about the bin ranges and number of bins you want to in the histogram before plotting
nBins    = 50
bin_lo   = 0
bin_high = 500

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("pt", "Muon veto pT[GeV]", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # get the Muon veto Pt
    VetoPt = tree[sample].cscRechitClusterMuonVetoPt
    
    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    #
    # Apply the cluster and event selections onto VetoPt
    VetoPt = VetoPt[sel_clusters][sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    VetoPt = ak.fill_none(ak.firsts(VetoPt,axis=1),-1 ) 
    
    h.fill(sample=sample,pt=VetoPt)

## Solution: muon veto pT distribution

The same logic as the jet veto: the cluster mask and then the event mask are applied to the veto pT, and the first cluster per event is kept.

Binning: 50 bins from 0 to 500 GeV, 10 GeV wide. The 20 GeV threshold used later is then a bin edge. Muon pT can go well above 500 GeV, and those entries overflow and are not shown.

Physics: the log-scale y axis (1e-5 to 0.1) reflects that most clusters have a muon veto pT near zero. A non-zero value means a reconstructed muon is matched to the cluster.

Pitfall: the variable name in the file is `cscRechitClusterMuonVetoPt`, which is the name used in the code. The exercise text writes it without `Pt`.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-5,0.1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies Muon veto pT < 20 GeV?

In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 20)
    print("sample = %s, eff(Muon VetoPT < 20) = %.5f "%(sample, eff))

## Solution: muon veto efficiency (pT < 20 GeV)

`integral_low(h, sample, 20)` is the fraction of clusters whose muon veto pT is below 20 GeV, that is, the clusters kept by the muon veto. The 20 GeV edge is on a 10 GeV bin boundary, so the number is exact.

Physics: a cluster matched to a high-pT muon is likely a muon bremsstrahlung shower, so the muon veto removes it. The signal is expected to have no matched muon, so its efficiency should stay high.

Pitfall: the veto threshold is a cut on the muon pT in GeV, not on the cluster size. Compare the efficiency for signal and background, not only the background rejection.

# Ex2 - plot number of ME11 and ME12 hits in the cluster

Additionally, punch-through jets or muon bremsstrahlung showers *might not get reconstructed as jets and muons*.

To fully remove these background, we remove clusters that have hits in the first CSC stations (ME11/ME12) that have little shielding in front.

In this exercise, you will plot the number of ME11/ME12 hits in clusters for signal and background and apply a selection on the two variables to remove clusters from jets and muons.

## plot number of ME11 and ME12 hits in cluster

 - Access the number of chamber hits in the cluster with the following variables:
   - `cscRechitClusterNRechitChamberPlus11`
   - `cscRechitClusterNRechitChamberMinus11`
   - `cscRechitClusterNRechitChamberPlus12`
   - `cscRechitClusterNRechitChamberMinus12`

In [ ]:
nBins    = 20
bin_lo   = 0
bin_high = 20

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("nME11_12", r"$N_{Me11/12 hits}$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Compute nME11_12
    nME11_12 = (tree[sample].cscRechitClusterNRechitChamberPlus11 + tree[sample].cscRechitClusterNRechitChamberMinus11
                + tree[sample].cscRechitClusterNRechitChamberPlus12 + tree[sample].cscRechitClusterNRechitChamberMinus12)
    
    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    nME11_12 = nME11_12[sel_clusters][sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    nME11_12 = ak.fill_none(ak.firsts(nME11_12,axis=1),-1 ) 
    
    h.fill(sample=sample,nME11_12=nME11_12)

## Solution: number of ME11/ME12 hits in the cluster

The four chamber-hit counts (ME+1/1, ME-1/1, ME+1/2, ME-1/2) are added per cluster to give one number, `nME11_12`. Then the same cluster and event masks are applied and the first cluster per event is kept.

Binning: 20 bins from 0 to 20 hits, so each integer count has its own bin. Counts above 20 are overflow and are not shown.

Units: dimensionless (number of hits). Pitfall: the four variables must be summed per cluster, so the sum has the same jagged structure as the mask. Adding them after taking the first cluster would mix up clusters.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-4,1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for nME11_12 ==0?

In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 1)
    print("sample = %s, eff(nME11_12 ==0) = %.5f "%(sample, eff))

## Solution: efficiency of nME11_12 == 0

The bins are one hit wide with edges at the integers, so bin 0 contains exactly the clusters with zero hits. `integral_low(h, sample, 1)` integrates up to the edge at 1, which is that bin alone. The no-cluster entries (-1) are in underflow and are not included.

Physics: clusters from jets or muon brems have hits in the first CSC stations, which have little shielding in front of them. A signal cluster from an LLP decaying in the CSC should have none, so this cut keeps the signal and removes those backgrounds.

Pitfall: integer counts need integer bin edges. If the edges were at non-integer values, the zero count could be split between two bins and the efficiency would be wrong.

# Ex3 - plot cluster eta

After we removed punch-through jets and muon brems, we observed that there are a lot more backgorund in higer $\eta$ region, where the muon reconstruction efficiency is lower and more pileup particles are present to create clusters.

In this exercise you will plot the cluster $\eta$ for signal and background and apply a selection on the variable to remove clusters from high $\eta$ region.

In [ ]:
nBins    = 50
bin_lo   = -2.5
bin_high = 2.5

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("eta", r"cluster $\eta$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get cluster Eta
    clsEta = tree[sample].cscRechitClusterEta
    
    # Event and cluster selections (make sure we have a cluster [and matched to LLP in CSC] )
    clsEta = clsEta[sel_clusters][sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -999 (not -1, which lies inside the eta range) for events without clusters
    clsEta = ak.fill_none(ak.firsts(clsEta,axis=1),-999 ) 
    
    h.fill(sample=sample,eta=clsEta)

## Solution: cluster eta

The cluster eta is read per cluster and masked with the cluster and event selections, then the first cluster per event is kept, as in the other exercises.

Binning: 50 bins from -2.5 to 2.5, 0.1 wide. The edges at -2 and 2 sit on bin boundaries, so the |eta| < 2 window in the next question is exact.

Pitfall: the template fills events without a cluster with -1. That value lies inside the eta range, so it would appear as a spike at eta = -1 and bias the efficiency. The fill value is changed to -999, which lands in underflow and is dropped by the efficiency formula.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-4,10)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $abs(\eta)<2$ ?

In [ ]:
# Hint: use integral_low and integral

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 2) - integral_low(h, sample, -2)
    print(r'sample = %s, eff(abs($\eta$)<2 ==0) = %.5f '%(sample, eff))

## Solution: |eta| < 2 efficiency

`integral_low` gives the fraction of clusters below a threshold. The window `-2 < eta < 2` is therefore the fraction below +2 minus the fraction below -2. Both terms share the same denominator, and the underflow entries (no cluster, filled with -999) appear in both terms and cancel in the difference.

Physics: the cluster eta cut removes the high-eta region, where muon reconstruction is weaker and there are more pileup clusters.

Pitfall: a single `integral_low(h, sample, 2)` would count all negative eta clusters too, so it would not be a symmetric |eta| cut. Subtracting the lower edge is the correct window.

# Ex4 - plot cluster time (after vetos are applied)

The remaining background clusters after punch-through jet and muon brem showers from the main collision are removed, are from low pT particles in pileup events.

To verify this, you will plot the cluster time for signal and background, after applying the vetoes, in this exercise to check for any out-of-time pileup contributions in data.


In [ ]:
nBins    = 120
bin_lo   = -20
bin_high = 40

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("time", r"cluster time[ns]", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get Variables
    clsTime  = tree[sample].cscRechitClusterTime
    jetVeto  = tree[sample].cscRechitClusterJetVetoPt
    muonVeto = tree[sample].cscRechitClusterMuonVetoPt
    clsEta   = tree[sample].cscRechitClusterEta
    
    # Apply the following selections:
    # - cluster and event selections 
    # - JetVeto < 10 GeV
    # - MuonVeto < 20 GeV
    # - cluster eta < 2
    
    # Try running without Jet and muon Veto as well!
    sel_cls = sel_clusters & (jetVeto < 10) & (muonVeto < 20) & (abs(clsEta) < 2)
    clsTime = clsTime[sel_cls][sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -999 for events without clusters that satisfy the cluster cuts
    clsTime = ak.fill_none(ak.firsts(clsTime,axis=1),-999 ) 
    
    h.fill(sample=sample,time=clsTime)

## Solution: cluster time after the vetoes

The jet veto, muon veto and eta cut are combined into one mask `sel_cls` with the cluster selection. The same mask is then applied to the time, so every variable refers to the same clusters. The event mask `sel_ev` is applied last.

Units: cluster time is in ns, and the veto pT values are in GeV. The bins are 0.5 ns wide from -20 to 40 ns, so the in-time edges -5 and 12.5 ns fall on bin edges.

Physics: after the vetoes the remaining background should be mostly low-pT pileup. Those clusters are spread in time, so the time plot shows whether an out-of-time component is present.

Pitfall: `-999` is used for events without a cluster, so they land in underflow and are not plotted. Removing the vetoes (the comment asks you to try it) shows how many background clusters the vetoes remove.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $-5<t<12.5$ ns ?

In [ ]:
def integral_range(h,sample,thres_lo,thres_high):
    
    name = h.dense_axes()[0].name

    numer = h.integrate("sample",sample).integrate(name,slice(thres_lo,thres_high)).values()[()]
    denom = h.integrate("sample",sample).sum(name,overflow="over").values()[()]    

    return numer/denom

for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_range(h, sample, -5, 12.5)
    print(r'sample = %s, eff( -5 < t < 12.5 ns) = %.5f '%(sample, eff))

## Solution: in-time efficiency for -5 < t < 12.5 ns

`integral_range(h, sample, -5, 12.5)` returns the fraction of clusters whose time lies inside the in-time window. The numerator integrates the window and the denominator sums the whole sample.

Physics: signal clusters from the in-time bunch crossing should sit inside the window, while background from pileup is spread over a wider time range. The efficiency tells us how much of the signal is kept by the in-time requirement.

Pitfall: both edges must be bin edges. With 0.5 ns bins from -20 to 40 ns, -5 and 12.5 are edges, so the window is exact. A 1 ns binning would cut the bin that contains 12.5 in the wrong place.

# Q: Can you suggest an out-of-time validation region based on the above plot?

A good validation region needs to have:
 - No (or at least small) signal conteminations (i.e. signal depleted) 
 - Similar background composition as your signal-rich region

# Ex5 - plot $\Delta\phi\mathrm{(cluster, MET)}$ and $N_{hits}$ in Out-of-time Validation Region (final discriminating variables)


The final discriminating variables that we will use to extract the signal and estimate background are 
 - the number of hits in the cluster ($N_{\text{hits}}$) and
 - the azimuthal angle between the cluster and MET ( $\Delta\phi\text{(cluster, MET)}$).

The background estimation method will be described in more detail in the next episode.

In this exercise, we will just plot the distributions of the two variables, to understand the shape of the two variables.

## Plot data only in OOT region to avoid unblinding!

## Implement the OOT selection you came up with above!

## plot $N_{hits}$

In [ ]:
nBins    = 50
bin_lo   = 0
bin_high = 250

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("nhits", r"$N_{hits}$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get Variables
    nhits    = tree[sample].cscRechitClusterSize
    clsTime  = tree[sample].cscRechitClusterTime
    jetVeto  = tree[sample].cscRechitClusterJetVetoPt
    muonVeto = tree[sample].cscRechitClusterMuonVetoPt
    clsEta   = tree[sample].cscRechitClusterEta

    
    # Only plot Out-of-Time(OOT) region for data
    if isData:
        sel_time = clsTime< -12.5
    else:
        sel_time = clsTime> -12.5   
        
    # Apply the following selections:
    # - cluster and event selections 
    # - JetVeto < 10 GeV
    # - MuonVeto < 20 GeV
    # - cluster eta < 2
    # - OOT selection for data; in-time selection for data
    sel_cls = sel_clusters & sel_time & (jetVeto < 10) & (muonVeto < 20) & (abs(clsEta) < 2)
    nhits = nhits[sel_cls][sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -1 for events without clusters that satisfy the cluster cuts
    nhits = ak.fill_none(ak.firsts(nhits,axis=1),-1 ) 
    
    h.fill(sample=sample,nhits=nhits)

## Solution: N_hits in the OOT region

`nhits` is `cscRechitClusterSize`, the number of CSC rechits in the cluster. The same cluster and event masks are applied, together with the time window, so the histogram shows the signal and background shapes in the validation region.

Blinding: in data only the out-of-time side (`t < -12.5 ns`) is filled, so the in-time signal region is not looked at. MC is filled on the in-time side for comparison.

The -12.5 ns boundary follows `ABCD_validation.ipynb`. The in-time window quoted in the efficiency question is -5 to 12.5 ns, so check that the two notebooks use the same boundary.

Binning: 50 bins from 0 to 250 hits, 5 hits wide. The thresholds 60, 130 and 190 used later sit on bin edges.

Pitfall: `fill_none(..., -1)` puts events without a cluster into underflow. Keep that in mind when reading the first bin.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
ax.set_yscale("log")
ax.set_ylim(1e-4,0.1)
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $N_{hits}>130$ ?


In [ ]:
for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral(h, sample, 130)
    print(r'sample = %s, eff($N_{hits}>130$) = %.5f '%(sample, eff))

## Solution: N_hits >= 130 efficiency

`integral(h, sample, 130)` returns the fraction of clusters with at least 130 hits. The question writes `>130`. For integer counts the ABCD notebook uses `>=`, so we keep that convention. Bins are 5 hits wide, so 130 is an edge and the number is exact.

Physics: a large number of hits is what a long, dense MDS shower looks like, so this cut keeps signal-like clusters.

Pitfall: when comparing with the ABCD scan, use the same `>=` convention. Otherwise the numbers differ by the events with exactly 130 hits.

## plot abs($\Delta\phi\mathrm{(cluster, MET)}$)

In [ ]:
nBins    = 64
bin_lo   = 0
bin_high = 3.2

h = hist.Hist("Events",
              hist.Cat("sample","sample"), 
              hist.Bin("dphi", r"$\Delta\phi(cls,MET)$", nBins, bin_lo, bin_high)
             )


for i, sample in enumerate(tree.keys()):
    
    #See if we are treating data
    isData = True if ("data" in sample) else False
    
    # build the selection arrays
    sel_clusters, sel_jet, sel_ev = select_events(tree[sample],isData)

    # Get Variables
    dphi     = np.abs(tree[sample].cscRechitClusterMet_dPhi)
    nhits    = tree[sample].cscRechitClusterSize
    clsTime  = tree[sample].cscRechitClusterTime
    jetVeto  = tree[sample].cscRechitClusterJetVetoPt
    muonVeto = tree[sample].cscRechitClusterMuonVetoPt
    clsEta   = tree[sample].cscRechitClusterEta

    
    # Only plot Out-of-Time(OOT) region
    if isData:
        sel_time = clsTime< -12.5
    else:
        sel_time = clsTime> -12.5
        
    # Apply the following selections:
    # - cluster and event selections 
    # - JetVeto < 10 GeV
    # - MuonVeto < 20 GeV
    # - cluster eta < 2
    # - OOT selection for data; in-time selection for data
    
    sel_cls = sel_clusters & sel_time & (jetVeto < 10) & (muonVeto < 20) & (abs(clsEta) < 2)
    dphi = dphi[sel_cls][sel_ev]
    
    # Select the VetoPt of the largest cluster 
    # fill_none puts -999 for events without clusters that satisfy the cluster cuts
    dphi = ak.fill_none(ak.firsts(dphi,axis=1),-999 ) 
    
    h.fill(sample=sample,dphi=dphi)

## Solution: |dphi(cluster, MET)| distribution in the OOT region

`dphi` is the azimuthal separation between the cluster and the MET. Only its size matters, so `np.abs` is applied. On the jagged array this keeps the one-value-per-cluster structure, and the same cluster and event masks are used as in the other exercises.

The region is chosen by time. Data are filled only out of time (`t < -12.5 ns`), so the in-time (blinded) region is not looked at. MC is filled in time (`t > -12.5 ns`) for comparison. The vetoes and the eta cut go into the same mask `sel_cls`.

Binning: 64 bins from 0 to 3.2 rad, 0.05 rad wide. Pi is about 3.14, so the range covers every value. The 0.75 cut and the scan range 0.45 to 0.95 fall on bin edges.

Pitfall: `cscRechitClusterMet_dPhi` is in radians, not degrees.

In [ ]:
ax = hist.plot1d(h,density=True)
ax.set_ylabel("Fraction of events / bin width")
hep.cms.label(ax=ax,label="Preliminary") 

# Q: What are the efficiencies for $abs(\Delta\phi(cls,MET)<0.75)$ ?


In [ ]:
for i, sample in enumerate(tree.keys()):
    # Fill code here
    eff = integral_low(h, sample, 0.75)
    print(r'sample = %s, eff($abs(\Delta\phi(cls,MET)<0.75)$) = %.5f '%(sample, eff))

## Solution: |dphi| < 0.75 efficiency

`integral_low(h, sample, 0.75)` gives the fraction of clusters with |dphi| below 0.75 rad. The histogram stores |dphi| in 0.05 rad bins, so 0.75 sits on a bin edge and the cut is exact.

Physics: the ABCD method uses this value to split the plane into a signal-like region (small |dphi|, since the signal cluster is expected to lie close in phi to the MET direction) and a background-like region.

Pitfall: the cut must be on the absolute value |dphi|. Without the absolute value, clusters on the negative side of phi would be counted as large |dphi| and be lost.